# Instant zoom with a tile pyramid

A raw Parquet scan reads every row on each pan or zoom. A **tile pyramid** stores Datashader's reduction state (counts, sums, min, max) at several resolutions, built once, so each viewport reads only the few small tiles it needs.

Every pyramid level is identical to a native `ds.Canvas(...).points(...)` aggregate at that resolution: counts, min and max bit for bit, sums and means up to floating-point summation order.

This notebook builds a small pyramid inside the browser, then explores it interactively. For real data, build the pyramid once with the CLI and host the folder next to your site:

```bash
datashader-browser build-tiles data.parquet --x x --y y --max-level 6 --out tiles/
```

In [ ]:
import numpy as np
import pandas as pd
import datashader as ds
from datashader_browser import TilePyramid, build_pyramid

rng = np.random.default_rng(0)
n = 2_000_000
centers = rng.normal(0, 3, size=(20, 2))
which = rng.integers(0, len(centers), n)
frame = pd.DataFrame({
    "x": centers[which, 0] + rng.normal(0, 0.3, n),
    "y": centers[which, 1] + rng.normal(0, 0.3, n),
    "value": rng.gamma(2, 2, n),
})

def chunks(df, size=250_000):
    for start in range(0, len(df), size):
        yield df.iloc[start:start + size]

pyramid = build_pyramid(
    chunks(frame), "/tmp/tiles", "x", "y",
    x_range=(-12, 12), y_range=(-12, 12),
    max_level=4, tile_size=256, values=["value"],
)
pyramid.manifest["binned_rows"], {k: len(v) for k, v in pyramid.manifest["levels"].items()}

Interactive exploration: each zoom reads only the intersecting tiles. Zooming past the deepest level would fall back to a raw scan if a `fallback=ParquetSource(...)` is given.

In [ ]:
from datashader_browser.holoviews import datashade

datashade(pyramid, agg=ds.mean("value"), width=700, height=500)

To open a pyramid hosted on a static site (GitHub Pages, S3, any web server), pass its URL. Only `manifest.json` and the visible tiles are downloaded:

```python
pyramid = TilePyramid("https://example.org/my-dataset/tiles")
datashade(pyramid)
```